# 07 -- Model sprawnosci turbiny

Ten notebook przedstawia modele turbin wodnych dla EW:

1. **Katalog turbin** -- Kaplan, Francis, smiglowa, przeplywowa
2. **Krzywe sprawnosci** -- eta_t = f(Q/Q_design) dla kazdego typu
3. **Wymiarowanie wirnika** -- srednica D1, obroty n z parametrow jednostkowych Q11, n11
4. **Macierz wyboru** -- nsN i D1 w funkcji (par biegunow, liczby turbin) z rekomendacja typu
5. **Powierzchnia sprawnosci** -- wielomianowa aproksymacja eta(Q11, n11) dla turbiny PL 10
6. **Sprawdzenie kawitacji** -- wspolczynnik Thomy, maksymalna wysokosc ssawna
7. **Zakres stosowalnosci H-Q** -- ktore turbiny pasuja do naszego przypadku
8. **Praca wieloturbinowa** -- rozdzial przeplywu na N turbin

Modele zaimplementowane w module `src/turbine.py`.

---

## Modul `src/turbine.py`

**Prompt do LLM tworzacy ten modul:**
> *"Stworz modul src/turbine.py do modelowania turbin wodnych.
> Modul powinien zawierac: (1) dataclass TurbineType z parametrami turbiny
> (Q11, n11, krzywa sprawnosci, zakres pracy), (2) katalog TURBINE_CATALOG
> z typami Kaplan/Francis/smiglowa/przeplywowa + modele PL10/PL20 z arkusza Excel,
> (3) turbine_efficiency(Q_ratio, type) -- interpolacja krzywej sprawnosci,
> (4) runner_diameter, rotational_speed, synchronous_speed, specific_speed --
> wymiarowanie z praw podobienstwa, (5) filter_applicable_turbines(H, Q),
> (6) dispatch_flow -- rozdzial przeplywu na N identycznych turbin."*

**Funkcje w module:**
- `TurbineType` -- dataclass z parametrami turbiny
- `TURBINE_CATALOG` -- slownik z 6 typami turbin
- `turbine_efficiency(Q_ratio, type)` -- sprawnosc z interpolacji krzywej
- `runner_diameter(Q_design, H, type)` -- srednica wirnika
- `rotational_speed(H, D1, type)` -- predkosc obrotowa
- `synchronous_speed(n_raw)` -- najblizsze obroty synchroniczne
- `specific_speed(n, P, H)` -- wyroznik szybkobieznosci nsN
- `filter_applicable_turbines(H, Q)` -- filtruj pasujace typy
- `dispatch_flow(Q_avail, Q_design, n_turb, type)` -- rozdzial przeplywu

## Konfiguracja

**Prompt do LLM:**
> *"Napisz kod konfiguracji notebooka: zaladuj modul src/turbine (TurbineType, TURBINE_CATALOG,
> turbine_efficiency, runner_diameter, rotational_speed, synchronous_speed, specific_speed,
> filter_applicable_turbines, dispatch_flow). Wyswietl liczbe typow w katalogu."*

**Uzyte moduly/funkcje:** `src.turbine`: TurbineType, TURBINE_CATALOG, turbine_efficiency, runner_diameter, rotational_speed, synchronous_speed, specific_speed, filter_applicable_turbines, dispatch_flow

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.turbine import (
    TurbineType,
    TURBINE_CATALOG,
    turbine_efficiency,
    runner_diameter,
    rotational_speed,
    synchronous_speed,
    specific_speed,
    filter_applicable_turbines,
    dispatch_flow,
)

print(f'Zaladowano katalog z {len(TURBINE_CATALOG)} typami turbin.')

Zaladowano katalog z 6 typami turbin.


---
## Krok 1: Katalog turbin

| Typ | Regulacja | Zakres H [m] | eta_peak | Zalety |
|-----|-----------|-------------|----------|--------|
| **Kaplan** | podwojna (lopaty + kierownica) | 1--20 | 0.92 | Plaska krzywa sprawnosci, szeroki zakres pracy |
| **Francis** | kierownica | 10--350 | 0.93 | Najwyzsza sprawnosc szczytkowa, duze spady |
| **Smiglowa** | brak (stale lopaty) | 1--15 | 0.90 | Tansza od Kaplana, ale waski zakres |
| **Przeplywowa** | -- | 1--200 | 0.84 | Prosta konstrukcja, dobra przy malych Q |

**Prompt do LLM:**
> *"Wyswietl katalog turbin jako tabelke z parametrami: nazwa, Q11, n11,
> eta_peak, zakres H, min/max Q_ratio."*

**Uzyte funkcje:** `src.turbine.TURBINE_CATALOG`

In [2]:
rows = []
for name, t in TURBINE_CATALOG.items():
    rows.append({
        'Typ': t.name_pl,
        'Q11': t.Q11,
        'n11': t.n11,
        'eta_peak': t.eta_peak,
        'H_min [m]': t.H_range[0],
        'H_max [m]': t.H_range[1],
        'Q_ratio_min': t.Q_ratio_min,
        'Q_ratio_max': t.Q_ratio_max,
    })
pd.DataFrame(rows)

,Typ,Q11,n11,eta_peak,H_min [m],H_max [m],Q_ratio_min,Q_ratio_max
0,Kaplan (podwojnie regulowana),1.20,158.0,0.920,1.0,20.0,0.15,1.1
1,Francis,0.80,120.0,0.930,10.0,350.0,0.40,1.0
2,Smiglowa (stale lopaty),1.05,133.0,0.900,1.0,15.0,0.65,1.0
3,Przeplywowa (Banki-Michell),0.45,40.0,0.840,1.0,200.0,0.10,1.0
4,PL 10 (Kaplan),1.20,158.0,0.876,1.0,20.0,0.15,1.0
5,PL 20 (Kaplan),1.05,133.0,0.900,1.0,20.0,0.20,1.0


---
## Krok 2: Krzywe sprawnosci

Sprawnosc turbiny zalezy od **obciazenia wzglednego** $Q/Q_{design}$:

$$\eta_t = f\left(\frac{Q}{Q_{design}}\right)$$

Kazdy typ turbiny ma inna charakterystyke:
- **Kaplan**: plaska krzywa -- wysoka sprawnosc w szerokim zakresie (15--110% Q_design)
- **Francis**: ostre maksimum -- najwyzsza szczytowa, ale szybko spada ponizej 60%
- **Smiglowa**: bardzo strome opadanie ponizej 80% -- waski zakres pracy
- **Przeplywowa**: umiarkowana, stabilna od 10% do 100%

**Ponizej Q_min turbina nie moze pracowac** (kawitacja, niestabilnosc).

**Prompt do LLM:**
> *"Narysuj krzywe sprawnosci eta(Q/Q_design) dla 4 glownych typow turbin
> na jednym wykresie. Uzyj plotly, zakres Q_ratio 0-1.2."*

**Uzyte funkcje:** `src.turbine.turbine_efficiency()`

In [3]:
Q_ratio = np.linspace(0.0, 1.2, 300)

colors = {
    'kaplan': 'royalblue', 'francis': 'firebrick',
    'propeller': 'green', 'crossflow': 'darkorange',
}
main_types = ['kaplan', 'francis', 'propeller', 'crossflow']

fig = go.Figure()
for name in main_types:
    t = TURBINE_CATALOG[name]
    eta = turbine_efficiency(Q_ratio, t)
    # Show only where eta > 0
    eta_plot = np.where(eta > 0, eta, np.nan)
    fig.add_trace(go.Scatter(
        x=Q_ratio, y=eta_plot,
        mode='lines', name=t.name_pl,
        line=dict(color=colors[name], width=2.5),
    ))
    # Mark data points
    fig.add_trace(go.Scatter(
        x=list(t.eta_curve_q), y=list(t.eta_curve_eta),
        mode='markers', showlegend=False,
        marker=dict(color=colors[name], size=6),
    ))

fig.update_layout(
    title='Krzywe sprawnosci turbin -- eta_t(Q/Q_design)',
    xaxis_title='Q / Q_design [-]',
    yaxis_title='Sprawnosc eta_t [-]',
    yaxis_range=[0, 1.0],
    height=500, hovermode='x unified',
)
fig.show()

### Modele PL10 i PL20 (z arkusza Excel)

Modele PL10 i PL20 to konkretne turbiny smiglowe/Kaplana z katalogu producenta
(parametry z arkusza WPE_2.xlsm):

**Prompt do LLM:**
> *"Porownaj krzywe sprawnosci modeli PL10 i PL20 z ogolna krzywa Kaplana
> na jednym wykresie. Wyswietl jako linie ciagle, Kaplan ogolny przerywany."*

**Uzyte funkcje:** `src.turbine.TURBINE_CATALOG`, `src.turbine.turbine_efficiency()`

In [4]:
fig = go.Figure()
for name, color in [('pl10', 'royalblue'), ('pl20', 'firebrick'), ('kaplan', 'gray')]:
    t = TURBINE_CATALOG[name]
    eta = turbine_efficiency(Q_ratio, t)
    eta_plot = np.where(eta > 0, eta, np.nan)
    fig.add_trace(go.Scatter(
        x=Q_ratio, y=eta_plot,
        mode='lines', name=t.name_pl,
        line=dict(color=color, width=2.5, dash='dash' if name=='kaplan' else 'solid'),
    ))

fig.update_layout(
    title='Porownanie modeli PL10, PL20 z ogolna krzywa Kaplana',
    xaxis_title='Q / Q_design [-]', yaxis_title='eta_t [-]',
    yaxis_range=[0, 1.0], height=450, hovermode='x unified',
)
fig.show()

---
## Krok 3: Wymiarowanie wirnika

Z **praw podobienstwa** turbin wodnych:

$$Q = Q_{11} \cdot D_1^2 \cdot \sqrt{H}$$

$$n = n_{11} \cdot \frac{\sqrt{H}}{D_1}$$

Stad srednica wirnika:

$$D_1 = \sqrt{\frac{Q_{design}}{Q_{11} \cdot \sqrt{H}}}$$

gdzie:
- $Q_{11}$ -- przeplyw jednostkowy [m$^3$/s] (parametr turbiny)
- $n_{11}$ -- predkosc jednostkowa [obr/min] (parametr turbiny)
- $D_1$ -- srednica wirnika [m]
- $H$ -- spad [m]

Predkosc obrotowa musi byc **synchroniczna** z siecia ($n_{sync} = 60f/p$).

**Prompt do LLM:**
> *"Napisz kod ktory dla H=6m i Q_design=11.7 m3/s (na turbine) obliczy
> srednice wirnika D1, obroty n, obroty synchroniczne i wyroznik szybkobieznosci
> dla kazdego typu turbiny z katalogu. Wyswietl tabelke."*

**Uzyte funkcje:** `runner_diameter()`, `rotational_speed()`, `synchronous_speed()`, `specific_speed()`

In [5]:
H = 6.0       # m
Q_des = 11.7  # m3/s (na turbine)
rho = 998
g = 9.81

rows = []
for name, t in TURBINE_CATALOG.items():
    if name == 'crossflow':
        continue  # turbina akcyjna (Banki-Michell) — nsN i obroty synchroniczne bez sensu
    D1 = runner_diameter(Q_des, H, t)
    n_raw = rotational_speed(H, D1, t)
    n_sync, poles = synchronous_speed(n_raw)
    P_kW = rho * g * Q_des * H * t.eta_peak / 1000
    # forma metryczna KM (x1.166) — spojna z macierza nsN (Krok 4) i kawitacja (Krok 6)
    nsN = specific_speed(n_sync, P_kW, H, metric_hp=True)
    rows.append({
        'Typ': t.name_pl,
        'D1 [m]': round(D1, 3),
        'n [obr/min]': round(n_raw, 1),
        'n_sync': round(n_sync, 0),
        'Biegunow': poles,
        'P [kW]': round(P_kW, 0),
        'nsN': round(nsN, 0),
    })

print(f'Wymiarowanie wirnika: H={H}m, Q_design={Q_des} m3/s')
pd.DataFrame(rows)

Wymiarowanie wirnika: H=6.0m, Q_design=11.7 m3/s


,Typ,D1 [m],n [obr/min],n_sync,Biegunow,P [kW],nsN
0,Kaplan (podwojnie regulowana),1.995,194.0,200.0,30,632.0,624.0
1,Francis,2.443,120.3,120.0,50,639.0,377.0
2,Smiglowa (stale lopaty),2.133,152.7,150.0,40,619.0,463.0
3,PL 10 (Kaplan),1.995,194.0,200.0,30,602.0,609.0
4,PL 20 (Kaplan),2.133,152.7,150.0,40,619.0,463.0


---
## Krok 4: Macierz wyboru -- wyroznik szybkobieznosci

**Wyroznik szybkobieznosci** $n_{sN}$ to klasyczny wskaznik wyboru typu turbiny:

$$n_{sN} = n \cdot \dfrac{\sqrt{P}}{H^{5/4}}$$

Konwencja w arkuszu **WPE_2.xlsm**: P w kW, ale wynik mnozony przez **1.166** zeby
byc spojnym z historyczna definicja w **metrycznych KM** ($\sqrt{1.36} \approx 1.166$).
Tej formy uzywamy tutaj, dla zgodnosci liczb z arkuszem referencyjnym.

### Typowe zakresy nsN dla typow turbin

| nsN | Typ turbiny |
|-----|-------------|
| < 80 | Pelton (jednodyszowy) |
| 80-400 | Francis (wolnobiezna) |
| 400-700 | Francis (szybkobiezna) |
| 700-900 | Kaplan / smiglowa |
| > 900 | Turbina osiowa (bulb) |

### Macierz (par biegunow x liczba turbin)

Dla naszego punktu projektowego $(Q_{design}, H_{design})$ liczymy nsN dla wszystkich
kombinacji **par biegunow** (1-30) i **liczby turbin** (1-8).
Pary biegunow okreslaja predkosc synchroniczna: $n = 60 \cdot f / p$.

**Uwaga o oznaczeniach:** WPE_2 nazywa kolumne `Number of poles`, ale numerycznie
uzywa **par biegunow** (`= poles/2`). Wartosc `4` w WPE_2 to **4 pary biegunow = 8 biegunow**,
co daje 750 obr/min przy 50 Hz. Tutaj uzywamy poprawnej nazwy `par biegunow`.

**Prompt do LLM:**
> *"Dla naszego punktu projektowego (Q_design=46.8 m3/s, H_design=6m, P_total liczone
> z rho*g*Q*H*eta_peak) zbuduj macierz nsN o wymiarach par_biegunow (1-30) x liczba_turbin (1-8).
> Uzyj formy WPE_2: nsN = (60*f/par_biegunow) * sqrt(P_per_turbine_kW) / H^(5/4) * 1.166.
> Narysuj heatmape z adnotacjami liczbowymi. Os Y odwrocona (wiecej par biegunow ku dolowi)."*

**Uzyte funkcje:** `numpy` (sqrt, arange), `plotly.graph_objects.Heatmap`.

In [6]:
# Punkt projektowy (z Krok 3)
H_design = 6.0      # spad projektowy [m]
Q_total = 46.8      # calkowity przeplyw projektowy [m3/s]
f_grid = 50         # czestotliwosc sieci [Hz]
eta_peak = TURBINE_CATALOG['kaplan'].eta_peak
rho, g = 998, 9.81

pole_pairs = np.arange(1, 31)              # 1..30 par biegunow
n_turbines_range = np.arange(1, 9)         # 1..8 turbin

# Macierz: wiersze = pary biegunow, kolumny = liczba turbin
P_per = rho * g * (Q_total / n_turbines_range) * H_design * eta_peak / 1000  # kW na turbine
n_sync = 60 * f_grid / pole_pairs                                            # rpm

# nsN macierzowo: (pole_pairs x n_turbines)
nsN_matrix = (n_sync[:, None] * np.sqrt(P_per[None, :]) / H_design**(5/4) * 1.166)

# Wykres heatmapy z adnotacjami
fig = go.Figure(go.Heatmap(
    z=nsN_matrix, x=n_turbines_range, y=pole_pairs,
    text=np.round(nsN_matrix).astype(int), texttemplate='%{text}',
    colorscale='Viridis',
    colorbar=dict(title='nsN'),
    hovertemplate='par biegunow: %{y}<br>turbin: %{x}<br>nsN = %{z:.0f}<extra></extra>',
))
fig.update_layout(
    title=f'Macierz nsN (WPE_2, x1.166): Q={Q_total} m3/s, H={H_design} m',
    xaxis_title='Liczba turbin', yaxis_title='Pary biegunow (poles/2)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(dtick=1),
    height=700,
)
fig.show()

print(f'Zakres nsN: {nsN_matrix.min():.0f} (max par biegunow, max turbin) ... '
      f'{nsN_matrix.max():.0f} (min par biegunow, 1 turbina)')

Zakres nsN: 221 (max par biegunow, max turbin) ... 18734 (min par biegunow, 1 turbina)


### Macierz srednic wirnika D1 (dla PL 10)

**Prompt do LLM:**
> *"Macierz D1 dla turbiny PL 10 (Q11=1.2). Wzor: D1 = sqrt(Q_per / Q11 / sqrt(H)).
> Rysuj heatmape rownolegle z macierza nsN — pomaga zobaczyc, ktore konfiguracje
> sa fizycznie realistyczne (D1 zbyt duze lub zbyt male sa drogie)."*

**Uzyte funkcje:** `numpy.sqrt`, `plotly.graph_objects.Heatmap`.

In [7]:
Q11_pl10 = TURBINE_CATALOG['pl10'].Q11
Q_per_matrix = Q_total / n_turbines_range          # m3/s na turbine
D1_matrix = np.sqrt(Q_per_matrix / Q11_pl10 / np.sqrt(H_design))  # nie zalezy od par biegunow!
# Rozszerz do macierzy o tym samym ksztalcie co nsN (powtarzamy wzdluz wierszy)
D1_full = np.broadcast_to(D1_matrix, nsN_matrix.shape).copy()

fig = go.Figure(go.Heatmap(
    z=D1_full, x=n_turbines_range, y=pole_pairs,
    text=np.round(D1_full, 2), texttemplate='%{text:.2f}',
    colorscale='Plasma',
    colorbar=dict(title='D1 [m]'),
    hovertemplate='turbin: %{x}<br>D1 = %{z:.2f} m<extra></extra>',
))
fig.update_layout(
    title=f'Macierz srednic wirnika D1 dla turbiny PL 10 (Q11={Q11_pl10})',
    xaxis_title='Liczba turbin', yaxis_title='Pary biegunow (powtorzone — D1 nie zalezy od n)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(dtick=1), height=500,
)
fig.show()

print(f'D1 dla PL 10 (Q11={Q11_pl10}): {D1_matrix.min():.2f}..{D1_matrix.max():.2f} m')

D1 dla PL 10 (Q11=1.2): 1.41..3.99 m


### Macierz rekomendacji typu turbiny

Dla kazdej kombinacji (par biegunow x liczba turbin) sugerujemy typ turbiny
wg zakresu nsN. **Wyszarzamy** te typy, ktore nie pasuja do naszego $H_{design}$
(wykluczenie po `H_range` z `TURBINE_CATALOG`).

**Prompt do LLM:**
> *"Klasifikuj kazda komorke macierzy nsN do jednego z 5 typow (Pelton/Francis-slow/
> Francis-fast/Kaplan/Bulb). Dla typow, ktorych zakres H wyklucza nasz H_design,
> oznacz komorke jako 'nieodpowiedni' (szary). Rysuj kategoryczna heatmape."*

**Uzyte funkcje:** `TURBINE_CATALOG` (do filtra H_range), `numpy.where`, `plotly.graph_objects.Heatmap`.

In [8]:
# Klasifikacja: nsN -> typ
def nsN_to_type(nsN_val):
    if nsN_val < 80:      return 'pelton'
    elif nsN_val < 400:   return 'francis_slow'
    elif nsN_val < 700:   return 'francis_fast'
    elif nsN_val < 900:   return 'kaplan'
    else:                  return 'bulb'

# Mapowanie typu na: katalogowy turbiny_typ (do filtra H), kod numeryczny, label, kolor
TYPE_INFO = {
    'pelton':       ('pelton',     0, 'Pelton',          '#9C27B0'),
    'francis_slow': ('francis',    1, 'Francis (wolna)', '#E91E63'),
    'francis_fast': ('francis',    2, 'Francis (szybka)','#FF5722'),
    'kaplan':       ('kaplan',     3, 'Kaplan',          '#3F51B5'),
    'bulb':         ('kaplan',     4, 'Bulb / osiowa',   '#009688'),  # podobny zakres do Kaplan
    'unfit':        (None,        -1, 'Nieodpowiedni',  '#BDBDBD'),
}

def is_H_fit(turb_key):
    """Czy H_design miesci sie w zakresie tego typu turbiny z katalogu."""
    if turb_key not in TURBINE_CATALOG:
        return True   # brak danych — nie wykluczamy
    H_min, H_max = TURBINE_CATALOG[turb_key].H_range
    return H_min <= H_design <= H_max

code_matrix = np.zeros_like(nsN_matrix, dtype=int)
label_matrix = np.empty_like(nsN_matrix, dtype=object)
for i in range(nsN_matrix.shape[0]):
    for j in range(nsN_matrix.shape[1]):
        t = nsN_to_type(nsN_matrix[i, j])
        turb_key, code_val, label, _ = TYPE_INFO[t]
        if turb_key is None or is_H_fit(turb_key):
            code_matrix[i, j] = code_val
            label_matrix[i, j] = label
        else:
            code_matrix[i, j] = -1
            label_matrix[i, j] = 'Nieodpowiedni\n(H poza zakresem)'

# Build discrete colorscale
all_codes = sorted({info[1] for info in TYPE_INFO.values()})
code_to_color = {info[1]: info[3] for info in TYPE_INFO.values()}
vmin, vmax = min(all_codes), max(all_codes)
colorscale = [[(c - vmin) / (vmax - vmin), code_to_color[c]] for c in all_codes]

fig = go.Figure(go.Heatmap(
    z=code_matrix, x=n_turbines_range, y=pole_pairs,
    text=label_matrix, texttemplate='%{text}', textfont=dict(size=9, color='white'),
    colorscale=colorscale, zmin=vmin, zmax=vmax, showscale=False,
    hovertemplate=('par biegunow: %{y}<br>turbin: %{x}<br>nsN '
                   '<extra>%{text}</extra>'),
))
fig.update_layout(
    title=f'Rekomendacja typu turbiny (po nsN i H_design={H_design} m)',
    xaxis_title='Liczba turbin', yaxis_title='Pary biegunow',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(dtick=1), height=700,
)
fig.show()

# Lista konfiguracji feasable (Kaplan w naszym H=6m)
feasible = []
for i, pp in enumerate(pole_pairs):
    for j, nt in enumerate(n_turbines_range):
        if code_matrix[i, j] == TYPE_INFO['kaplan'][1]:  # Kaplan-fit
            feasible.append((pp, int(nt), int(60*f_grid/pp), nsN_matrix[i, j]))
feasible_df = pd.DataFrame(feasible, columns=['par_biegunow','n_turbin','n_sync_rpm','nsN']).head(10)
print(f'Pierwsze 10 konfiguracji Kaplan w naszym H={H_design}m:')
print(feasible_df.to_string(index=False))

Pierwsze 10 konfiguracji Kaplan w naszym H=6.0m:
 par_biegunow  n_turbin  n_sync_rpm        nsN
            8         7         375 885.082950
            8         8         375 827.919290
            9         6         333 849.776755
            9         7         333 786.740400
            9         8         333 735.928258
           10         5         300 837.795416
           10         6         300 764.799080
           10         7         300 708.066360
           11         4         272 851.530682
           11         5         272 761.632196


---
## Krok 5: Powierzchnia sprawnosci PL 10 (przyblizenie wielomianowe)

Producenci turbin podaja **karte charakterystyk** (turbine map) — sprawnosc $\eta$
jako funkcje punktow pracy w zmiennych jednostkowych $(Q_{11}, n_{11})$.
Karta to tabela wartosci, ale do obliczen wygodnie jest **dopasowac wielomian**.

W arkuszu **WPE_2.xlsm** uzyto wielomianu 3. stopnia (rozdzielnego, bez czlonow krzyzowych)
ktory aproksymuje karte turbiny **PL 10**:

$$\eta[\%] = 14.1094 + 0.02554 \cdot G + 0.838 \cdot n_{11}$$
$$\hphantom{\eta[\%]} - 1.234\cdot 10^{-5}\cdot G^2 - 3.620\cdot 10^{-3}\cdot n_{11}^2$$
$$\hphantom{\eta[\%]} + 1.159\cdot 10^{-9}\cdot G^3 + 4.200\cdot 10^{-6}\cdot n_{11}^3$$

gdzie $G = Q_{11} \cdot 1000$ (przeplyw jednostkowy w **l/s**, nie m³/s) i $n_{11}$ w rpm.

**Uwaga:** wielomian jest **rozdzielny** w $G$ i $n_{11}$ — brak czlonu $G\cdot n_{11}$.
To uproszczenie: w rzeczywistosci sprawnosc zalezy od **wzajemnego** ustawienia
kierownicy (Q) i obrotow (n). Wielomian pasuje dla turbiny PL 10 w typowym zakresie,
ale nie ekstrapolauje dobrze poza ten zakres.

**Prompt do LLM:**
> *"Zdefiniuj funkcje pl10_eta(Q11, n11) implementujaca wielomian z WPE_2.
> Narysuj jej powierzchnie na siatce Q11 in (0.05..1.5 m3/s) i n11 in (100..250 rpm)
> jako wykres konturowy. Zaznacz punkt projektowy PL 10 (Q11=1.2, n11=158)."*

**Uzyte funkcje:** `numpy.meshgrid`, `plotly.graph_objects.Contour`.

In [9]:
def pl10_eta(Q11, n11):
    """Sprawnosc turbiny PL 10 jako funkcja (Q11, n11). Wielomian z WPE_2.xlsm.

    Args:
        Q11: przeplyw jednostkowy [m3/s]
        n11: predkosc jednostkowa [rpm]
    Returns:
        sprawnosc [-] (0..1)
    """
    G = np.asarray(Q11) * 1000  # m3/s -> l/s (konwencja Excela)
    E = np.asarray(n11)
    eta_pct = (14.1094114
               + 0.0255361978 * G + 0.838004938 * E
               - 0.0000123420439 * G**2 - 0.00361969217 * E**2
               + 0.00000000115932973 * G**3 + 0.00000420032345 * E**3)
    return eta_pct / 100.0

# Walidacja w punkcie projektowym
pl10 = TURBINE_CATALOG['pl10']
eta_check = pl10_eta(pl10.Q11, pl10.n11)
print(f'PL 10 w punkcie projektowym (Q11={pl10.Q11}, n11={pl10.n11}):')
print(f'  Wielomian: eta = {eta_check:.4f}')
print(f'  Katalog:   eta_peak = {pl10.eta_peak:.4f}')

# Siatka do wykresu konturowego
Q11_grid = np.linspace(0.05, 1.5, 80)
n11_grid = np.linspace(100, 250, 80)
QQ, NN = np.meshgrid(Q11_grid, n11_grid)
ETA = pl10_eta(QQ, NN)
ETA = np.clip(ETA, 0, 1)  # wielomian moze ekstrapolowac poza [0,1]

fig = go.Figure(go.Contour(
    z=ETA * 100, x=Q11_grid, y=n11_grid,
    colorscale='Viridis',
    contours=dict(start=0, end=90, size=5, showlabels=True,
                  labelfont=dict(size=10, color='white')),
    colorbar=dict(title='η [%]'),
))
fig.add_trace(go.Scatter(
    x=[pl10.Q11], y=[pl10.n11],
    mode='markers+text', marker=dict(symbol='star', size=18, color='red'),
    text=[f'PL 10 design<br>η={pl10.eta_peak:.0%}'],
    textposition='top right', textfont=dict(color='red'),
    name='Punkt projektowy', showlegend=False,
))
fig.update_layout(
    title='Karta charakterystyk PL 10 (wielomianowa aproksymacja z WPE_2.xlsm)',
    xaxis_title='Q11 [m3/s]', yaxis_title='n11 [rpm]',
    height=550,
)
fig.show()

print()
print('Uwaga: ten wielomian nie ma czlonu krzyzowego Q11*n11,')
print('wiec linie stalej sprawnosci nie pokazuja sprzezenia Q-n (wielomian separowalny).')
print('To uproszczenie WPE_2; rzeczywiste karty turbin maja silne sprzezenie Q-n.')

PL 10 w punkcie projektowym (Q11=1.2, n11=158.0):
  Wielomian: eta = 0.8759
  Katalog:   eta_peak = 0.8760



Uwaga: ten wielomian nie ma czlonu krzyzowego Q11*n11,
wiec linie stalej sprawnosci nie pokazuja sprzezenia Q-n (wielomian separowalny).
To uproszczenie WPE_2; rzeczywiste karty turbin maja silne sprzezenie Q-n.


---
## Krok 6: Sprawdzenie kawitacji (wspolczynnik Thomy)

Kawitacja to lokalne wrzenie wody przy spadku cisnienia ponizej cisnienia
nasycenia pary — niszczy lopaty wirnika w ciagu **miesiecy**, nie lat.
Aby ja wykluczyc, projektant sprawdza wspolczynnik Thomy:

$$\sigma = \dfrac{H_{atm} - H_v - H_s}{H_{net}}$$

gdzie:
- $H_{atm}$ — cisnienie atmosferyczne wyrazone jako slup wody [m] (~10.3 m przy poziomie morza),
- $H_v$ — cisnienie pary wody [m] (~0.12 m przy 10°C),
- $H_s$ — **wysokosc ssawna** (rzedna srodka wirnika nad poziomem wody dolnej; dodatnia = wirnik nad lustrem, ujemna = zatopiony),
- $H_{net}$ — spad netto.

Aby uniknac kawitacji: $\sigma > \sigma_{krytyczny}$, gdzie $\sigma_{kr}$ zalezy od **wyroznika szybkobieznosci** $n_{sN}$ (im wieksza, tym bardziej wirnik jest narazony):

| Typ turbiny | $\sigma_{kr}(n_{sN})$ |
|-------------|------------------------|
| Pelton | $0$ (turbina akcyjna — bez kawitacji) |
| Francis | $7.54 \cdot 10^{-5} \cdot n_{sN}^{1.41}$ |
| Kaplan / smiglowa | $4.41 \cdot 10^{-9} \cdot n_{sN}^{2.81}$ |

(formuly z Penche 2004 i Czekalski 2008; nsN w konwencji **metrycznych KM**, z czynnikiem 1.166 — patrz Krok 4).

**Maksymalna wysokosc ssawna** (gorny limit zamontowania wirnika):

$$H_{s,max} = H_{atm} - H_v - \sigma_{kr} \cdot H_{net}$$

Jesli $H_{s,max} < 0$, wirnik musi byc **zatopiony** ponizej poziomu wody dolnej.

**Prompt do LLM:**
> *"Dla naszego punktu projektowego (H=6m, P_per_turbina, n_sync z Krok 3) policz nsN
> (forma metryczna), sigma_critical i H_s_max dla turbin pasujacych do H=6m (Kaplan, smiglowa).
> Wyswietl tabele i krotka rekomendacje."*

**Uzyte funkcje:** `src.turbine.specific_speed(metric_hp=True)`, `thoma_sigma_critical`,
`suction_head_max`, `atmospheric_pressure_head`, `vapor_pressure_head`.

In [10]:
from src.turbine import (
    thoma_sigma_critical, suction_head_max,
    atmospheric_pressure_head, vapor_pressure_head,
)

# Punkt projektowy (z Krok 3)
H = 6.0
Q_des = 11.7        # m3/s na turbine
P_per_kw = 998 * 9.81 * Q_des * H * TURBINE_CATALOG['kaplan'].eta_peak / 1000  # ~632 kW

# Warunki lokalne (Malczyce ~120 m n.p.m., woda Odry srednio ~10°C)
ELEVATION_M = 120
WATER_TEMP_C = 10

H_atm = atmospheric_pressure_head(ELEVATION_M)
H_v = vapor_pressure_head(WATER_TEMP_C)
print(f'Warunki lokalne: H_atm={H_atm:.2f} m, H_v={H_v:.3f} m')

# Sprawdzenie dla turbin pasujacych do H=6m
rows = []
for name, t in TURBINE_CATALOG.items():
    if name == 'crossflow':
        continue  # turbina akcyjna — kawitacja (Thoma sigma) jej nie dotyczy
    H_min, H_max = t.H_range
    if not (H_min <= H <= H_max):
        continue  # pomin nieodpowiednie do naszego H
    D1 = float(np.sqrt(Q_des / (t.Q11 * np.sqrt(H))))
    n_raw = t.n11 * np.sqrt(H) / D1
    n_sync, poles = synchronous_speed(n_raw, mode='nearest')
    P_kw = 998 * 9.81 * Q_des * H * t.eta_peak / 1000  # moc wg eta_peak danego typu
    nsN_hp = specific_speed(n_sync, P_kw, H, metric_hp=True)

    try:
        sigma_c = thoma_sigma_critical(nsN_hp, name)
    except ValueError:
        continue
    H_s_max = suction_head_max(nsN_hp, H, name, elevation_m=ELEVATION_M, water_temp_C=WATER_TEMP_C,
                               safety_margin=0.10)  # zapas projektowy Δσ
    rows.append({
        'Typ': t.name_pl,
        'n_sync [rpm]': round(n_sync, 0),
        'nsN (HP)': round(nsN_hp, 0),
        'sigma_kr': round(sigma_c, 3),
        'H_s_max [m]': round(H_s_max, 2),
        'Zatopiony?': 'NIE' if H_s_max > 0 else f'TAK (min. {-H_s_max:.2f} m ponizej WD)',
    })

df_cav = pd.DataFrame(rows)
print(f'\nKawitacja dla H={H}m, Q_des={Q_des} m3/s, P={P_per_kw:.0f} kW na turbine:')
df_cav

Warunki lokalne: H_atm=10.20 m, H_v=0.124 m

Kawitacja dla H=6.0m, Q_des=11.7 m3/s, P=632 kW na turbine:


,Typ,n_sync [rpm],nsN (HP),sigma_kr,H_s_max [m],Zatopiony?
0,Kaplan (podwojnie regulowana),200.0,624.0,0.316,7.58,NIE
1,Smiglowa (stale lopaty),150.0,463.0,0.137,8.65,NIE
2,PL 10 (Kaplan),200.0,609.0,0.295,7.70,NIE
3,PL 20 (Kaplan),150.0,463.0,0.137,8.65,NIE


---
## Krok 7: Zakres stosowalnosci H-Q

Kazdy typ turbiny ma zakres spadow H i przeplywow Q w ktorym moze pracowac.
Ten wykres pomaga w wstepnym doborze typu.

**Prompt do LLM:**
> *"Narysuj wykres stosowalnosci H-Q: dla kazdego typu turbiny zaznacz obszar
> (H_min-H_max) jako poziome pasy. Zaznacz nasz punkt projektowy (H=6m)."*

**Uzyte funkcje:** `src.turbine.filter_applicable_turbines()`

In [11]:
# Wykres stosowalnosci H-Q
main_types = ['kaplan', 'francis', 'propeller', 'crossflow']
colors_map = {'kaplan': 'royalblue', 'francis': 'firebrick',
              'propeller': 'green', 'crossflow': 'darkorange'}

fig = go.Figure()
for i, name in enumerate(main_types):
    t = TURBINE_CATALOG[name]
    H_min, H_max = t.H_range
    fig.add_trace(go.Bar(
        y=[t.name_pl], x=[H_max - H_min], base=[H_min],
        orientation='h', name=t.name_pl,
        marker_color=colors_map[name], opacity=0.7,
        text=f'{H_min}-{H_max} m', textposition='inside',
    ))

fig.add_vline(x=6.0, line_dash='dash', line_color='black',
    annotation_text='H=6m (nasz przypadek)')

fig.update_layout(
    title='Zakres stosowalnosci turbin wg spadu H',
    xaxis_title='Spad H [m]', xaxis_type='log',
    height=350, showlegend=False, barmode='overlay',
)
fig.show()

# Filtruj pasujace
valid = filter_applicable_turbines(H=6.0, Q_design=50)
print(f'Turbiny pasujace dla H=6m: {[TURBINE_CATALOG[k].name_pl for k in valid]}')

Turbiny pasujace dla H=6m: ['Kaplan (podwojnie regulowana)', 'Smiglowa (stale lopaty)', 'PL 10 (Kaplan)', 'PL 20 (Kaplan)']


---
## Krok 8: Praca wieloturbinowa -- rozdzial przeplywu

Przy wielu identycznych turbinach dostepny przeplyw rozdzielamy tak,
aby kazda dzialajaca jednostka byla **jak najblizej swojego punktu projektowego** (najlepsza sprawnosc):

1. Liczba aktywnych turbin: $n_{active} = \min\big(n_{turbines},\; \max(1,\; \lceil Q / Q_{design} \rceil)\big)$
2. Przeplyw na turbine: $Q_{per} = \min(Q / n_{active},\; Q_{design})$
3. Jesli przy podziale na $n$ turbin kazda spadlaby ponizej $Q_{min}$ (a $Q \geq Q_{design}$) — uruchamiamy **mniej** turbin przy $Q_{design}$, nadmiar na przelew (fallback). Dopiero gdy $Q_{per} < Q_{min}$ przy $n_{active}=1$ — elektrownia stoi.

**Zaleta wielu turbin:** wieksza moc zainstalowana (mozemy 'lapac' wieksze przeplywy)
i mniej wody traconej na przelewie przy szczytach.

**Prompt do LLM:**
> *"Porownaj prace 1 turbiny vs 2 turbin vs 4 turbin dla tego samego calkowitego
> Q_design. Pokaz jak sprawnosc zmienia sie w funkcji Q/Q_total_design.
> Uzyj turbiny Kaplana."*

**Uzyte funkcje:** `src.turbine.dispatch_flow()`, `src.turbine.turbine_efficiency()`

In [12]:
Q_total_design = 46.8  # m3/s (calkowity przeplyw projektowy EW)
kaplan = TURBINE_CATALOG['kaplan']
Q_avail = np.linspace(0.5, Q_total_design * 1.1, 400)

configs = [(1, 'gray'), (2, 'royalblue'), (4, 'firebrick')]
results = {}
for n_turb, _ in configs:
    Q_per_des = Q_total_design / n_turb           # Q_design *per turbine*
    Q_min_per = kaplan.Q_ratio_min * Q_per_des    # Q_min *per turbine*
    Q_each, n_act = dispatch_flow(Q_avail, Q_per_des, n_turb, kaplan)
    # eta per running turbine (0 gdy n_act == 0 — plant stoi)
    Q_ratio = np.where(Q_per_des > 0, Q_each / Q_per_des, 0.0)
    eta = turbine_efficiency(Q_ratio, kaplan)
    eta = np.where(n_act > 0, eta, 0.0)            # twardy zero gdy plant stoi
    # Calkowita moc plantu (kW) — pelny lancuch energetyczny
    rho, g, H = 998, 9.81, 6.0; eta_g = 0.96  # eta_g stala (placeholder) — pelny model generatora w nb 08
    P_plant_kW = n_act * rho * g * Q_each * H * eta * eta_g / 1000.0
    results[n_turb] = dict(
        Q_per_des=Q_per_des, Q_min_per=Q_min_per,
        Q_each=Q_each, n_act=n_act, eta=eta, P_kW=P_plant_kW,
    )

# 3-panel plot: eta_per_turbine, n_active (step), total P_kW
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[
        'Sprawnosc turbiny eta_t (per dzialajaca turbina)',
        'Liczba dzialajacych turbin n_active',
        'Calkowita moc elektrowni P [kW]',
    ], vertical_spacing=0.08)

x_pct = Q_avail / Q_total_design * 100
for n_turb, color in configs:
    r = results[n_turb]
    label = f'{n_turb} turbin (Q_per={r["Q_per_des"]:.1f} m3/s, Q_min={r["Q_min_per"]:.1f} m3/s)'
    # eta — None gdy n_act==0 (zeby nie laczyc kresek przez zero)
    eta_plot = np.where(r['n_act'] > 0, r['eta'], np.nan)
    fig.add_trace(go.Scatter(x=x_pct, y=eta_plot, mode='lines',
        name=label, line=dict(color=color, width=2.5),
        legendgroup=str(n_turb)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_pct, y=r['n_act'], mode='lines',
        line=dict(color=color, width=2, shape='hv'), showlegend=False,
        legendgroup=str(n_turb)), row=2, col=1)
    fig.add_trace(go.Scatter(x=x_pct, y=r['P_kW'], mode='lines',
        line=dict(color=color, width=2.5), showlegend=False,
        legendgroup=str(n_turb)), row=3, col=1)
    # Adnotacja punktu uruchomienia (gdy Q_avail == Q_min_per — 1 turbina rusza)
    Q_min_pct = r['Q_min_per'] / Q_total_design * 100
    fig.add_vline(x=Q_min_pct, line=dict(color=color, width=1, dash='dot'),
        opacity=0.4, row=1, col=1)

fig.update_xaxes(title_text='Q dostepne / Q_total_design [%]', row=3, col=1)
fig.update_yaxes(title_text='eta_t [-]', range=[0, 1], row=1, col=1)
fig.update_yaxes(title_text='n_active', range=[-0.2, max(c[0] for c in configs)+0.5], row=2, col=1)
fig.update_yaxes(title_text='P [kW]', row=3, col=1)
fig.update_layout(height=800, hovermode='x unified',
    title='Dyspozytor wieloturbinowy — Kaplan, Q_total_design=46.8 m3/s, H=6 m')
fig.show()

print('Klucze dla zrozumienia wykresow:')
print('  - Q_min_per_turbine = Q_ratio_min * Q_design_per_turbine')
print('    Wiecej turbin → mniejszy Q_design na turbine → mniejszy Q_min → plant rusza wczesniej.')
print('  - Skok eta przy Q = Q_design_per_turbine: dispatch przelacza n_active 1→2 itd.,')
print('    co rozklada przeplyw na wiecej turbin → kazda dziala blizej polowy mocy (mniejsza eta).')
print('  - Eta == 0 gdy Q_avail < Q_min_per_turbine: plant fizycznie stoi.')
print('  - Calkowita moc P_plant rosnie skokowo gdy uruchamia sie kolejna turbina.')

Klucze dla zrozumienia wykresow:
  - Q_min_per_turbine = Q_ratio_min * Q_design_per_turbine
    Wiecej turbin → mniejszy Q_design na turbine → mniejszy Q_min → plant rusza wczesniej.
  - Skok eta przy Q = Q_design_per_turbine: dispatch przelacza n_active 1→2 itd.,
    co rozklada przeplyw na wiecej turbin → kazda dziala blizej polowy mocy (mniejsza eta).
  - Eta == 0 gdy Q_avail < Q_min_per_turbine: plant fizycznie stoi.
  - Calkowita moc P_plant rosnie skokowo gdy uruchamia sie kolejna turbina.


---
## Podsumowanie

W tym notebooku przedstawiono:

| Model | Opis | Funkcja / Komorka |
|-------|------|--------------------|
| Katalog turbin | 6 typow z parametrami | `TURBINE_CATALOG` |
| Sprawnosc | eta_t(Q/Q_design) z interpolacji | `turbine_efficiency()` |
| Wymiarowanie | D1, n z Q11/n11 | `runner_diameter()`, `rotational_speed()` |
| Macierz nsN | (pary biegunow x n turbin) wg WPE_2 | inline `nsN_matrix` |
| Powierzchnia eta PL 10 | wielomian z WPE_2 | inline `pl10_eta()` |
| **Kawitacja** | Thoma σ, max wysokosc ssawna | `thoma_sigma_critical()`, `suction_head_max()` |
| Stosowalnosc H-Q | filtrowanie po H | `filter_applicable_turbines()` |
| Wieloturbinowa praca | rozdzial Q na N turbin | `dispatch_flow()` |

**Kluczowe wnioski:**
- Kaplan najlepsza dla niskich spadow (H=6m) — szeroki zakres pracy
- Wiecej turbin pozwala uzyc wiekszej czesci przeplywu w okresie szczytu
- Francis nie pasuje do H=6m (wymaga H>=10m)
- Wielomian PL 10 z WPE_2 jest **rozdzielny w Q11 i n11** (uproszczenie); w realnych
  kartach turbin sprzezenie Q-n jest silne
- Z zapasem projektowym (Δσ=0.10) nawet przy niskim spadzie wirnik ustawia sie blisko poziomu
  wody dolnej (lekko nad/pod nim); przy wyzszym nsN/spadzie turbina musi byc zatopiona

**Dalej:** notebook 08 — model sprawnosci generatora